In [1]:
from Model import Conformer
from Metrics import metrics

import numpy as np
import torch
import torch.nn as nn
from scipy.signal import resample
from sklearn.metrics import precision_score, recall_score, f1_score, cohen_kappa_score
import os
import shutil
import pickle
import time

torch.manual_seed(42)
torch.cuda.manual_seed(42)

class Trainer:
	def __init__(self, config):
		self.config = config
		self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
		self.model = Conformer(config['emb_size'], config['depth'], config['num_classes']).to(self.device).to(torch.float64)

		self.optimizer = torch.optim.Adam(self.model.parameters(), lr=config['learning_rate'], betas=(config['b1'], config['b2']))
		self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(self.optimizer, 'min')
		self.criterion = nn.CrossEntropyLoss()

		os.makedirs('Results', exist_ok=True)
	
	def z_score_um(self, train_dataset, test_dataset, eps=1e-6):
		train_data = torch.stack([train_dataset[i][0] for i in range(len(train_dataset))])
		train_mean = train_data.mean(dim=(0,2), keepdim=True)
		train_std = train_data.std(dim=(0,2),keepdim=True)
		train_data = (train_data - train_mean) / (train_std + eps)
		train_labels = torch.stack([train_dataset[i][1] for i in range(len(train_dataset))])
		train_dataset = torch.utils.data.TensorDataset(train_data, train_labels)

		eval_data = torch.stack([test_dataset[i][0] for i in range(len(test_dataset))])
		eval_data = (eval_data - train_mean) / (train_std + eps)
		eval_labels = torch.stack([test_dataset[i][1] for i in range(len(test_dataset))])
		test_dataset = torch.utils.data.TensorDataset(eval_data, eval_labels)

		return train_dataset, test_dataset
	
	def train_epoch(self, epoch, train_loader):
		epoch += 1
		self.model.train()
		total_correct, total_loss = 0, 0

		for batch_eeg, batch_labels in train_loader:
			batch_eeg = batch_eeg.to(self.device, dtype=torch.float64)
			batch_labels = batch_labels.to(self.device)
			
			Tok, outputs = self.model(batch_eeg)
			cls_loss = self.criterion(outputs, batch_labels)
			self.optimizer.zero_grad()
			cls_loss.backward()
			self.optimizer.step()
			
			preds = torch.argmax(outputs, dim=1)
			total_correct += (preds == batch_labels).sum().item()
			total_loss += cls_loss.item()

		train_loss = total_loss / len(train_loader)
		train_acc = total_correct / len(train_loader.dataset)
		
		return train_loss, train_acc
	
	def evaluate_epoch(self, eval_loader):
		self.model.eval()
		total_loss, total_correct = 0, 0
		all_preds, all_labels = [], []
		
		with torch.no_grad():
			for eval_eeg, eval_labels in eval_loader:
				eval_eeg = eval_eeg.to(self.device, dtype=torch.float64)
				eval_labels = eval_labels.to(self.device)
				
				Tok, outputs = self.model(eval_eeg)
				cls_loss = self.criterion(outputs, eval_labels)
				total_loss += cls_loss.item()
				
				preds = torch.max(outputs, 1)[1]
				total_correct += (preds == eval_labels).sum().item()
				
				all_preds.extend(preds.cpu().numpy())
				all_labels.extend(eval_labels.cpu().numpy())
		
		loss = total_loss / len(eval_loader)
		acc = total_correct / len(eval_loader.dataset)
		precision = precision_score(all_labels, all_preds, zero_division=0)
		recall = recall_score(all_labels, all_preds)
		f1 = f1_score(all_labels, all_preds)
		kappa = cohen_kappa_score(all_labels, all_preds)
		
		return loss, acc, precision, recall, f1, kappa

	def train_subject(self, subject, mode):
		if mode == 0:
			data_root = config['mi_root'] + str(subject) + '.pkl'
		elif mode == 1:
			data_root = config['ma_root'] + str(subject) + '.pkl'
		elif mode == 2:
			data_root = config['wg_root'] + str(subject) + '.pkl'
		with open(data_root, 'rb') as f:
			data = pickle.load(f)
		
		eeg = data['eeg']
		labels = data['labels']
		
		if mode == 0 or mode == 1:
			# eeg = resample(eeg, num=1000, axis=-1)
			# eeg = torch.tensor(eeg, dtype=torch.float64)
			eeg = eeg.unsqueeze(1)
		elif mode == 2:
			if config['downsample']:
				eeg = eeg[:, :, 1::4]
			eeg = np.expand_dims(eeg, axis=1)
			eeg = torch.tensor(eeg, dtype=torch.float64)
			labels = torch.tensor(labels, dtype=torch.long)
		eeg, labels = eeg.to(self.device), labels.to(self.device)
		print(f'EEG: {eeg.shape} Labels: {labels.shape}')
		
		train_size = int(config['ratio'] * len(eeg)) # training/testing ratio
		eval_size = len(eeg) - train_size
		
		dataset = torch.utils.data.TensorDataset(eeg, labels)
		train_dataset, eval_dataset = torch.utils.data.random_split(dataset, [train_size, eval_size])
		if config['z_score']:
			train_dataset, eval_dataset = self.z_score_um(train_dataset, eval_dataset)
		train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=self.config['batch_size'], shuffle=True)
		eval_loader = torch.utils.data.DataLoader(eval_dataset, batch_size=self.config['batch_size'], shuffle=False)
		
		acc_list, precision_list, recall_list, f1_list, kappa_list = [], [], [], [], []
		
		for epoch in range(self.config['num_epochs']):
			train_loss, train_acc = self.train_epoch(epoch, train_loader)
			eval_loss, eval_acc, precision, recall, f1, kappa = self.evaluate_epoch(eval_loader)
			
			acc_list.append(eval_acc)
			precision_list.append(precision)
			recall_list.append(recall)
			f1_list.append(f1)
			kappa_list.append(kappa)
			
			print(f"Subject {subject} | Epoch {epoch+1}/{self.config['num_epochs']} | "
				f"Train Loss: {train_loss:.2f} | Train Acc: {train_acc:.2f} | "
				f"Eval Loss: {eval_loss:.2f} | Eval Acc: {eval_acc:.2f}")
			
		metrics(subject, config['log_name'], config['log_mode'], acc_list, precision_list, recall_list, f1_list, kappa_list)
		return

# Hyperparameters
# the correlation between Q, K, V and embedding size
config = {
	'emb_size': 64,
	'depth': 4,
	'num_classes': 2,
	'batch_size': 16,
	'num_epochs': 200,
	'learning_rate': 1e-3,
	'b1': 0.5,
	'b2': 0.999,
	'ratio': 0.6,
	'num_run': 3,
	'mode': 0,
	'log_name': '101',
	'log_mode': 1,
	'z_score': True,
	'downsample': True,
	'mi_root': '../../Dataset/EF-MI-MA/EF-PKL-MI/',
	'ma_root': '../../Dataset/EF-MI-MA/EF-PKL-MA/',
	'wg_root': '../../Dataset/EF-WG/WG/',
}

# Initialize and run trainer
current_dir = os.getcwd()
results_dir = os.path.join(current_dir, 'Results', config['log_name'])
file_name = 'SD.ipynb'

source_dir = os.path.join(current_dir, file_name)
destination_dir = results_dir

for run in range(config['num_run']):
	trainer = Trainer(config)
	mode = config['mode']
	num_subject = (29 if mode == 0 or mode == 1 else 26) # MI = 0, MA = 1, WG = 2
	print(f'Run {run+1}', time.asctime(time.localtime(time.time())))
	for subject in range(num_subject):
		subject += 1
		print(f"\n=== Subject {subject} ===")
		results = trainer.train_subject(subject, mode)
	print(time.asctime(time.localtime(time.time())))

os.makedirs(destination_dir, exist_ok=True)
shutil.copy2(source_dir, destination_dir)

/data_main/junlin/conda_env/mbi/lib/python3.10/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Run 1 Sat Jun 21 19:03:42 2025

=== Subject 1 ===
EEG: torch.Size([60, 1, 30, 4000]) Labels: torch.Size([60])
Subject 1 | Epoch 1/200 | Train Loss: 0.95 | Train Acc: 0.50 | Eval Loss: 0.76 | Eval Acc: 0.46
Subject 1 | Epoch 2/200 | Train Loss: 0.87 | Train Acc: 0.33 | Eval Loss: 0.81 | Eval Acc: 0.46
Subject 1 | Epoch 3/200 | Train Loss: 0.72 | Train Acc: 0.58 | Eval Loss: 0.73 | Eval Acc: 0.54
Subject 1 | Epoch 4/200 | Train Loss: 1.12 | Train Acc: 0.47 | Eval Loss: 0.83 | Eval Acc: 0.38
Subject 1 | Epoch 5/200 | Train Loss: 0.74 | Train Acc: 0.61 | Eval Loss: 1.19 | Eval Acc: 0.25
Subject 1 | Epoch 6/200 | Train Loss: 0.67 | Train Acc: 0.69 | Eval Loss: 1.15 | Eval Acc: 0.46
Subject 1 | Epoch 7/200 | Train Loss: 0.88 | Train Acc: 0.64 | Eval Loss: 1.95 | Eval Acc: 0.33
Subject 1 | Epoch 8/200 | Train Loss: 0.69 | Train Acc: 0.72 | Eval Loss: 1.53 | Eval Acc: 0.38
Subject 1 | Epoch 9/200 | Train Loss: 0.72 | Train Acc: 0.64 | Eval Loss: 1.54 | Eval Acc: 0.42
Subject 1 | Epoch 10/200 |

KeyboardInterrupt: 